# Semana 3 — Window Functions

Funções de janela em PySpark: `row_number`, `rank`, soma acumulada, `lag` e `lead`, usando dados sintéticos de pedidos.

Equivalentes do `OVER (PARTITION BY ... ORDER BY ...)` do SQL — preservam o detalhe linha a linha enquanto calculam sobre o grupo. Parte do plano de reposicionamento para engenharia de dados.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from decimal import Decimal
import random
from datetime import date, timedelta

random.seed(42)
produtos = ["VR", "VA", "VT", "AUXILIO"]
base = date(2026, 1, 1)

schema_pedidos = StructType([
    StructField("id_pedido",    IntegerType(),      False),
    StructField("id_cliente",   IntegerType(),      False),
    StructField("produto",      StringType(),       True),
    StructField("data_pedido",  DateType(),         True),
    StructField("valor_bruto",  DecimalType(12, 2), True),
    StructField("pct_desconto", DecimalType(6, 4),  True),
])

dados = [
    (
        i,
        random.randint(1000, 1050),
        random.choice(produtos),
        base + timedelta(days=random.randint(0, 300)),
        Decimal(str(round(random.uniform(50, 5000), 2))),
        Decimal(str(round(random.uniform(0, 0.15), 4))),
    )
    for i in range(1, 50001)
]

pedidos = spark.createDataFrame(dados, schema_pedidos)
pedidos.show(5)

In [0]:

janela = Window.partitionBy("produto").orderBy(F.col("valor_bruto").desc())
(
pedidos.withColumn("posicao", F.row_number().over(janela)) 
    .filter(F.col("produto") == "VR") 
    .select("produto", "id_pedido", "valor_bruto", "posicao") 
    .show(20)
)    

In [0]:
top3 = (
    pedidos
    .withColumn("posicao", F.row_number().over(janela))
    .filter(F.col("posicao") <= 3)
    .select("produto", "id_pedido", "valor_bruto", "posicao")
    .orderBy("produto", "posicao")
)

top3.show(12)